In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path("/Users/finn/Documents/research project/tox21.csv")
if not csv_path.exists():
    raise FileNotFoundError(f"{csv_path} not found")

# Load CSV
df = pd.read_csv(csv_path, sep=None, engine='python')# ...existing code...
matrix = df.to_numpy()
print("Matrix shape:", matrix.shape)
print("First 5 rows:\n", matrix[:5])
# ...existing code...)

# Keep only numeric columns (drop e.g. SMILES or other metadata if present)
numeric_df = df.select_dtypes(include=[np.number])
if numeric_df.shape[1] != df.shape[1]:
    print("Non-numeric columns detected and removed:", [c for c in df.columns if c not in numeric_df.columns])

# Convert to NumPy matrix
matrix = numeric_df.to_numpy()

# Basic checks / output
print("Loaded DataFrame shape:", df.shape)
print("Numeric matrix shape:", matrix.shape)
print("First 5 rows of matrix:\n", matrix[:5])

print(numeric_df.head())

Matrix shape: (7831, 14)
First 5 rows:
 [[0.0 0.0 1.0 nan nan 0.0 0.0 1.0 0.0 0.0 0.0 0.0 'TOX3021'
  'CCOc1ccc2nc(S(N)(=O)=O)sc2c1']
 [0.0 0.0 0.0 0.0 0.0 0.0 0.0 nan 0.0 nan 0.0 0.0 'TOX3020'
  'CCN1C(=O)NC(c2ccccc2)C1=O']
 [nan nan nan nan nan nan nan 0.0 nan 0.0 nan nan 'TOX3024'
  'CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]3CC[C@@]21C']
 [0.0 0.0 0.0 0.0 0.0 0.0 0.0 nan 0.0 nan 0.0 0.0 'TOX3027'
  'CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C']
 [0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 'TOX20800'
  'CC(O)(P(=O)(O)O)P(=O)(O)O']]
Non-numeric columns detected and removed: ['ID', 'X']
Loaded DataFrame shape: (7831, 14)
Numeric matrix shape: (7831, 12)
First 5 rows of matrix:
 [[ 0.  0.  1. nan nan  0.  0.  1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0. nan  0. nan  0.  0.]
 [nan nan nan nan nan nan nan  0. nan  0. nan nan]
 [ 0.  0.  0.  0.  0.  0.  0. nan  0. nan  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]]
   NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD

In [4]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

mol = Chem.MolFromSmiles("CCN1C(=O)NC(c2ccccc2)C1=O")
# 创建 Morgan 指纹生成器，等价于原来的 radius=2（ECFP4）
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fp = generator.GetFingerprint(mol)

print(fp.ToBitString())   # 可选：查看 0/1 字符串

# 预览 fp1 的前5个字符，并显示总长度
bitstr = fp.ToBitString()
print("前5个字符:", bitstr[:5])
print("总长度:", len(bitstr))
type(fp)
fp

0000010000000000000000000101000000100000000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001000000100000000000010000000000000000000000000000000000000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001000000000000000000010000000000000000000000000000000000000000010000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000010000000010000000000000000001000000000000001000000000000000000000000000000

In [5]:
# 将第14列SMILES用RDKit生成ECFP4指纹，并加到df第15列
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

# 获取SMILES列（第14列，索引13）
smiles_list = df.iloc[:, 13].astype(str)
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)


In [6]:

def smiles_to_fp(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fp = generator.GetFingerprint(mol)
        return fp.ToBitString()
    else:
        return '0' * 2048

# 生成指纹字符串
fp_list = smiles_list.apply(smiles_to_fp)
# 加到df第15列
fp_col_name = 'ECFP4'
df[fp_col_name] = fp_list
print(df[[df.columns[13], fp_col_name]].head())
print('ecfp4:', df[fp_col_name].shape)
print(df.shape)
print(df.head)

[15:42:36] WARNING: not removing hydrogen atom without neighbors


                                                   X  \
0                       CCOc1ccc2nc(S(N)(=O)=O)sc2c1   
1                          CCN1C(=O)NC(c2ccccc2)C1=O   
2  CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...   
3                    CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C   
4                          CC(O)(P(=O)(O)O)P(=O)(O)O   

                                               ECFP4  
0  0000000000000000000000000000000000000000000000...  
1  0000010000000000000000000101000000100000000000...  
2  0000000000010000000000000000000000000000000000...  
3  0100000000000000000000000000000000000000000000...  
4  0000000000000000000000000000000000000000000000...  
ecfp4: (7831,)
(7831, 15)
<bound method NDFrame.head of       NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  \
0       0.0        0.0     1.0           NaN    NaN        0.0            0.0   
1       0.0        0.0     0.0           0.0    0.0        0.0            0.0   
2       NaN        NaN     NaN    

**下面把ecfp列拆分成2048个列**

In [7]:
df_with_fp = df


In [8]:

# 将ecfp列拆分为2048个新列，每列为一位
fp_matrix = df_with_fp['ECFP4'].apply(lambda x: pd.Series(list(x))).astype(int)
fp_matrix.columns = [f'ecfp4_{i}' for i in range(2048)]
# 合并到原df
df_with_fp = pd.concat([df_with_fp.drop('ECFP4', axis=1), fp_matrix], axis=1)
print(df_with_fp.head())


   NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  \
0    0.0        0.0     1.0           NaN    NaN        0.0            0.0   
1    0.0        0.0     0.0           0.0    0.0        0.0            0.0   
2    NaN        NaN     NaN           NaN    NaN        NaN            NaN   
3    0.0        0.0     0.0           0.0    0.0        0.0            0.0   
4    0.0        0.0     0.0           0.0    0.0        0.0            0.0   

   SR-ARE  SR-ATAD5  SR-HSE  ...  ecfp4_2038  ecfp4_2039 ecfp4_2040  \
0     1.0       0.0     0.0  ...           0           0          0   
1     NaN       0.0     NaN  ...           0           0          0   
2     0.0       NaN     0.0  ...           0           0          0   
3     NaN       0.0     NaN  ...           0           0          0   
4     0.0       0.0     0.0  ...           0           0          0   

  ecfp4_2041  ecfp4_2042  ecfp4_2043  ecfp4_2044  ecfp4_2045  ecfp4_2046  \
0          0           0    

In [9]:
print('shape of df_with_fp:', df_with_fp.shape)
print(df_with_fp.columns[2061:])

shape of df_with_fp: (7831, 2062)
Index(['ecfp4_2047'], dtype='object')


**下面是mml**

In [10]:
from transformers import AutoTokenizer, AutoModel

local_path = "/Users/finn/Documents/research project/chemberta"         
tok = AutoTokenizer.from_pretrained(local_path)
model = AutoModel.from_pretrained(local_path)

/opt/anaconda3/envs/sklearn-vscode/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/anaconda3/envs/sklearn-vscode/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/opt/anaconda3/envs/sklearn-vscode/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/opt/anaconda3/envs/sklearn-vscode/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please u

In [11]:
print("Tokenizer vocab size:", tok.vocab_size)  
print("Model hidden size:", model.config.hidden_size) 

Tokenizer vocab size: 591
Model hidden size: 384


In [12]:
import torch
smiles = "CCO"
inputs = tok(smiles, return_tensors="pt")
with torch.no_grad():
    emb = model(**inputs).last_hidden_state.mean(1)
print(emb.shape)   
type(emb) 
emb      # 期望 torch.Size([1, 768])

torch.Size([1, 384])


tensor([[ 1.9905e-01, -4.1334e-01, -4.3834e-05, -1.0590e-01,  2.1656e-01,
          1.7400e-01, -2.3140e-01,  2.1255e-01,  2.5075e-01,  9.7360e-02,
          2.6469e-01,  1.5758e-01, -1.0781e-01, -1.7522e-01,  8.1607e-02,
         -6.0106e-03,  2.3697e-01,  5.9968e-02, -1.0953e-01, -7.5874e-02,
         -8.0191e-02,  5.4353e-02, -2.2580e-01,  3.8510e-01, -7.1495e-02,
          2.3202e-02, -2.6642e-01,  2.7034e-01, -1.1186e-01,  4.6111e-02,
          1.5230e-01,  8.2144e-02,  3.0375e-02, -2.4096e-01,  3.5307e-01,
          2.7099e-01,  2.0638e-01, -6.3178e-02, -3.0743e-01,  1.3885e-01,
         -2.3358e-01, -1.1718e-01,  2.8089e-01, -8.9062e-02,  7.3852e-02,
         -4.5625e-02,  4.0536e-01,  1.8908e-01,  1.2245e-02,  4.4745e-01,
         -2.0945e-01,  1.9200e-01,  3.2102e-01, -1.3366e-01,  1.8047e-01,
          2.2158e-01, -7.5090e-02, -3.7825e-01,  1.7098e-01, -1.0124e-01,
         -3.3934e-01, -2.3016e-01,  4.8730e-02, -1.5164e-01, -1.2607e-01,
          1.2813e-01, -1.5486e-01, -4.

**上面是llm的测试，下面是前五排测试**

In [13]:
df0 = df_with_fp
import torch, numpy as np
from transformers import AutoTokenizer, AutoModel

# 1. 复用你本地已加载的 384 维 ChemBERTa
local_path = "/Users/finn/Documents/research project/chemberta"
tok = AutoTokenizer.from_pretrained(local_path)
model = AutoModel.from_pretrained(local_path)

# 2. 384 维 embed 函数（你图中已验证）
@torch.no_grad()
def smi2vec(smi: str) -> np.ndarray:
    try:
        inputs = tok(smi, return_tensors="pt")
        hidden = model(**inputs).last_hidden_state   # [1, L, 384]
        return hidden.mean(1).squeeze().numpy()      # (384,)
    except Exception:
        return np.zeros(384, dtype=np.float32)

# 3. 用 SMILES 列（'X'）转成 384 维向量 → 最后一列
df0['chem_embed'] = df0['X'].apply(smi2vec)



Some weights of RobertaModel were not initialized from the model checkpoint at /Users/finn/Documents/research project/chemberta and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:

print(df0.tail())
#print('embedding列形状:', df0[emb_col_name].shape)
print('df0总形状:', df0.shape)

      NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  \
7826    NaN        NaN     NaN           NaN    NaN        NaN            NaN   
7827    1.0        1.0     0.0           0.0    1.0        0.0            NaN   
7828    1.0        1.0     0.0           0.0    1.0        1.0            0.0   
7829    1.0        1.0     0.0           NaN    1.0        1.0            0.0   
7830    0.0        0.0     NaN           0.0    0.0        0.0            0.0   

      SR-ARE  SR-ATAD5  SR-HSE  ...  ecfp4_2039  ecfp4_2040 ecfp4_2041  \
7826     0.0       NaN     0.0  ...           0           0          0   
7827     NaN       0.0     0.0  ...           0           0          0   
7828     1.0       0.0     0.0  ...           0           0          0   
7829     0.0       0.0     0.0  ...           0           0          0   
7830     0.0       0.0     0.0  ...           0           0          0   

     ecfp4_2042  ecfp4_2043  ecfp4_2044  ecfp4_2045  ecfp4_2046  ecf

In [15]:
# 查看chem_embed列的基本信息和第一行详细内容
print('chem_embed列类型:', type(df0['chem_embed']))
print('chem_embed列长度:', len(df0['chem_embed']))
print('第一行embedding shape:', np.array(df0['chem_embed'].iloc[0]).shape)
print('第一行embedding内容:')
print(df0.iloc[0, :20])
print(df0.iloc[0, 2062])



chem_embed列类型: <class 'pandas.core.series.Series'>
chem_embed列长度: 7831
第一行embedding shape: (384,)
第一行embedding内容:
NR-AR                                     0.0
NR-AR-LBD                                 0.0
NR-AhR                                    1.0
NR-Aromatase                              NaN
NR-ER                                     NaN
NR-ER-LBD                                 0.0
NR-PPAR-gamma                             0.0
SR-ARE                                    1.0
SR-ATAD5                                  0.0
SR-HSE                                    0.0
SR-MMP                                    0.0
SR-p53                                    0.0
ID                                    TOX3021
X                CCOc1ccc2nc(S(N)(=O)=O)sc2c1
ecfp4_0                                     0
ecfp4_1                                     0
ecfp4_2                                     0
ecfp4_3                                     0
ecfp4_4                                     0
ecfp4_5     

In [16]:
dfmtrx =  df0

In [17]:
# 将dfmtrx最后一列chem_embed拆分为384个新列，每列为一维float
emb_matrix = dfmtrx['chem_embed'].apply(lambda x: pd.Series(x)).astype(float)
emb_matrix.columns = [f'chemberta_{i}' for i in range(384)]
# 合并到原dfmtrx，去掉chem_embed列
final_df = pd.concat([dfmtrx.drop('chem_embed', axis=1), emb_matrix], axis=1)
print(final_df.head())
print('新final_df形状:', final_df.shape)


   NR-AR  NR-AR-LBD  NR-AhR  NR-Aromatase  NR-ER  NR-ER-LBD  NR-PPAR-gamma  \
0    0.0        0.0     1.0           NaN    NaN        0.0            0.0   
1    0.0        0.0     0.0           0.0    0.0        0.0            0.0   
2    NaN        NaN     NaN           NaN    NaN        NaN            NaN   
3    0.0        0.0     0.0           0.0    0.0        0.0            0.0   
4    0.0        0.0     0.0           0.0    0.0        0.0            0.0   

   SR-ARE  SR-ATAD5  SR-HSE  ...  chemberta_374  chemberta_375 chemberta_376  \
0     1.0       0.0     0.0  ...      -0.026885      -0.153081     -0.043036   
1     NaN       0.0     NaN  ...      -0.122446      -0.088287     -0.170991   
2     0.0       NaN     0.0  ...       0.134588       0.154017     -0.137180   
3     NaN       0.0     NaN  ...      -0.030880       0.012388      0.091946   
4     0.0       0.0     0.0  ...      -0.148199       0.090314      0.105764   

  chemberta_377  chemberta_378  chemberta_379  che

In [18]:

print('最后5列:', final_df.columns[-5:])
print('第一行embedding前20维:', final_df.iloc[0, -384:-364].values)

最后5列: Index(['chemberta_379', 'chemberta_380', 'chemberta_381', 'chemberta_382',
       'chemberta_383'],
      dtype='object')
第一行embedding前20维: [np.float64(-0.004681919235736132) np.float64(0.07996965944766998)
 np.float64(0.00023069977760314941) np.float64(-0.0015455385437235236)
 np.float64(-0.04665106534957886) np.float64(0.0836593359708786)
 np.float64(-0.15015047788619995) np.float64(0.06574404984712601)
 np.float64(-0.040835633873939514) np.float64(0.26852184534072876)
 np.float64(0.2336626648902893) np.float64(0.15678028762340546)
 np.float64(-0.003783488180488348) np.float64(-0.039731819182634354)
 np.float64(-0.09453804045915604) np.float64(0.03241746500134468)
 np.float64(0.0008918722742237151) np.float64(0.022209521383047104)
 np.float64(-0.05056031048297882) np.float64(0.20494228601455688)]


In [ ]:
# 保存final_df到本地文件夹
final_df.to_csv("/Users/finn/Documents/research project/final_df.csv", index=False)
print("final_df已保存到 final_df.csv")

final_df已保存到 final_df.csv


: 